In [1]:
# 03 - Silver: limpieza y unificación
import pandas as pd
import numpy as np
from pathlib import Path
from datetime import datetime, timezone
import json

# Auto-detección de entorno: Colab o local (VS Code)
IS_COLAB = False
try:
    from google.colab import drive  # type: ignore
    IS_COLAB = True
except Exception:
    IS_COLAB = False

if IS_COLAB:
    drive.mount('/content/drive')
    PROJECT_ROOT = Path('/content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2')
else:
    # En local se asume que este notebook vive en <repo>/notebooks
    PROJECT_ROOT = Path.cwd()
    if PROJECT_ROOT.name.lower() == 'notebooks':
        PROJECT_ROOT = PROJECT_ROOT.parent

BRONZE = PROJECT_ROOT / 'data/bronze'
SILVER = PROJECT_ROOT / 'data/silver'
SILVER.mkdir(parents=True, exist_ok=True)

BRONZE_MARKET = BRONZE / 'market_data'
BRONZE_MACRO = BRONZE / 'macro_data'
BRONZE_SENTIMENT = BRONZE / 'sentiment_data'

print('IS_COLAB:', IS_COLAB)
print('PROJECT_ROOT:', PROJECT_ROOT)
print('BRONZE:', BRONZE)
print('SILVER:', SILVER)

Mounted at /content/drive
IS_COLAB: True
PROJECT_ROOT: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2
BRONZE: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/bronze
SILVER: /content/drive/MyDrive/01. USFQ/01. Maestría IA/12. Trabajo de Titulación/05. V2/data/silver


In [2]:
# Utilidades
def read_csv_if_exists(path: Path):
    return pd.read_csv(path) if path.exists() else None

def clean_numeric(df: pd.DataFrame, cols):
    for c in cols:
        if c in df.columns:
            df[c] = pd.to_numeric(df[c], errors='coerce')
    return df

def clean_market(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df['timestamp'] = pd.to_datetime(df['timestamp'], utc=True, errors='coerce')
    df = clean_numeric(df, ['open', 'high', 'low', 'close', 'volume'])

    # Reglas básicas OHLC
    df = df.dropna(subset=['timestamp', 'open', 'high', 'low', 'close'])
    df = df[(df['high'] >= df['low']) & (df['high'] >= df['open']) & (df['high'] >= df['close'])]
    df = df[(df['low'] <= df['open']) & (df['low'] <= df['close'])]

    df = df.sort_values('timestamp').drop_duplicates(subset=['timestamp', 'interval'], keep='last')
    return df.reset_index(drop=True)

def clean_macro(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ts_col = 'timestamp' if 'timestamp' in df.columns else 'bucket_ts'
    df[ts_col] = pd.to_datetime(df[ts_col], utc=True, errors='coerce')
    df = clean_numeric(df, ['open', 'high', 'low', 'close', 'volume'])
    df = df.dropna(subset=[ts_col, 'close'])
    df = df.rename(columns={ts_col: 'timestamp'})
    df = df.sort_values('timestamp').drop_duplicates(subset=['timestamp', 'feature_name'], keep='last')
    return df.reset_index(drop=True)

def clean_sentiment(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    ts_col = 'bucket_ts' if 'bucket_ts' in df.columns else 'timestamp'
    df[ts_col] = pd.to_datetime(df[ts_col], utc=True, errors='coerce')
    num_cols = ['sentiment_compound_mean', 'sentiment_ordinal_mean', 'sentiment_ordinal_sum', 'mentions', 'sentiment_compound', 'sentiment_ordinal']
    df = clean_numeric(df, [c for c in num_cols if c in df.columns])
    df = df.dropna(subset=[ts_col])
    df = df.rename(columns={ts_col: 'timestamp'})
    return df.reset_index(drop=True)

def save_silver(df: pd.DataFrame, name: str, metadata: dict):
    csv_path = SILVER / f'{name}.csv'
    json_path = SILVER / f'{name}.json'
    df.to_csv(csv_path, index=False)
    with open(json_path, 'w', encoding='utf-8') as f:
        json.dump(metadata, f, ensure_ascii=False, indent=2, default=str)
    print(f'OK -> {csv_path.name} ({len(df):,} filas)')

In [3]:
# Cargar Bronze
market_candidates = [
    BRONZE_MARKET / 'btc_combined_data.csv',
    BRONZE_MARKET / 'binance_btc_1h.csv',
]
market_path = next((p for p in market_candidates if p.exists()), None)

macro_candidates = [
    BRONZE_MACRO / 'macro_combined_1d.csv',
]
macro_path = next((p for p in macro_candidates if p.exists()), None)

sent_candidates = [
    BRONZE_SENTIMENT / 'bitcoin_news_reddit_agg_1h.csv',
    BRONZE_SENTIMENT / 'bitcoin_news_reddit.csv',
]
sent_path = next((p for p in sent_candidates if p.exists()), None)

if market_path is None:
    raise FileNotFoundError('No se encontró archivo de mercado en Bronze.')
if macro_path is None:
    raise FileNotFoundError('No se encontró archivo macro en Bronze.')
if sent_path is None:
    raise FileNotFoundError('No se encontró archivo de sentimiento en Bronze.')

market_raw = pd.read_csv(market_path)
macro_raw = pd.read_csv(macro_path)
sent_raw = pd.read_csv(sent_path)

print('market_path:', market_path.name, '| filas:', len(market_raw))
print('macro_path:', macro_path.name, '| filas:', len(macro_raw))
print('sent_path:', sent_path.name, '| filas:', len(sent_raw))

market_path: btc_combined_data.csv | filas: 3000
macro_path: macro_combined_1d.csv | filas: 77745
sent_path: bitcoin_news_reddit_agg_1h.csv | filas: 200


In [4]:
# Limpieza Silver
market = clean_market(market_raw)
macro = clean_macro(macro_raw)
sent = clean_sentiment(sent_raw)

# Si viene sentimiento granular, agregamos a 1h
if 'sentiment_compound_mean' not in sent.columns and 'sentiment_compound' in sent.columns:
    group_keys = ['timestamp'] + (['subreddit'] if 'subreddit' in sent.columns else [])
    sent = (
        sent.groupby(group_keys, dropna=False)
        .agg(
            sentiment_compound_mean=('sentiment_compound', 'mean'),
            sentiment_ordinal_mean=('sentiment_ordinal', 'mean'),
            sentiment_ordinal_sum=('sentiment_ordinal', 'sum'),
            mentions=('content_id', 'count') if 'content_id' in sent.columns else ('sentiment_ordinal', 'count'),
        )
        .reset_index()
    )

print('Limpio market:', market.shape)
print('Limpio macro:', macro.shape)
print('Limpio sentiment:', sent.shape)

Limpio market: (3000, 9)
Limpio macro: (0, 10)
Limpio sentiment: (200, 6)


In [5]:
# Pivot macro a formato ancho (wide) usando close por feature
macro_wide = (
    macro[['timestamp', 'feature_name', 'close']]
    .dropna(subset=['feature_name'])
    .pivot_table(index='timestamp', columns='feature_name', values='close', aggfunc='last')
    .reset_index()
)

# Sentimiento agregado global (promedio entre subreddits por timestamp)
sent_cols = [c for c in ['sentiment_compound_mean', 'sentiment_ordinal_mean', 'sentiment_ordinal_sum', 'mentions'] if c in sent.columns]
sent_global = sent.groupby('timestamp', as_index=False)[sent_cols].mean()

# Separar mercado por frecuencia y unir
market_1h = market[market['interval'] == '1h'].copy()
market_4h = market[market['interval'] == '4h'].copy()
market_1d = market[market['interval'] == '1d'].copy()

silver_1h = market_1h.merge(sent_global, on='timestamp', how='left').sort_values('timestamp')
silver_1d = market_1d.merge(macro_wide, on='timestamp', how='left').merge(sent_global, on='timestamp', how='left').sort_values('timestamp')

# Relleno controlado
for df in [silver_1h, silver_1d]:
    for c in df.columns:
        if c not in ['timestamp', 'source', 'symbol', 'interval'] and pd.api.types.is_numeric_dtype(df[c]):
            df[c] = df[c].replace([np.inf, -np.inf], np.nan)
            df[c] = df[c].ffill(limit=24)

print('silver_1h:', silver_1h.shape)
print('silver_4h (solo limpio):', market_4h.shape)
print('silver_1d:', silver_1d.shape)

silver_1h: (1000, 13)
silver_4h (solo limpio): (1000, 9)
silver_1d: (1000, 13)


In [6]:
# Guardar salidas Silver
ts_now = datetime.now(timezone.utc).isoformat()

meta_1h = {
    'dataset': 'silver_market_sentiment_1h',
    'created_at_utc': ts_now,
    'rows': int(len(silver_1h)),
    'columns': list(silver_1h.columns),
    'sources': {
        'market': str(market_path.name),
        'sentiment': str(sent_path.name),
    }
}

meta_4h = {
    'dataset': 'silver_market_4h',
    'created_at_utc': ts_now,
    'rows': int(len(market_4h)),
    'columns': list(market_4h.columns),
    'sources': {'market': str(market_path.name)}
}

meta_1d = {
    'dataset': 'silver_market_macro_sentiment_1d',
    'created_at_utc': ts_now,
    'rows': int(len(silver_1d)),
    'columns': list(silver_1d.columns),
    'sources': {
        'market': str(market_path.name),
        'macro': str(macro_path.name),
        'sentiment': str(sent_path.name),
    }
}

save_silver(silver_1h, 'silver_market_sentiment_1h', meta_1h)
save_silver(market_4h, 'silver_market_4h', meta_4h)
save_silver(silver_1d, 'silver_market_macro_sentiment_1d', meta_1d)

OK -> silver_market_sentiment_1h.csv (1,000 filas)
OK -> silver_market_4h.csv (1,000 filas)
OK -> silver_market_macro_sentiment_1d.csv (1,000 filas)


In [7]:
# Verificación rápida
silver_files = sorted([p.name for p in SILVER.glob('*')])
print('--- SILVER FILES ---')
for f in silver_files:
    print(f)

display(silver_1h.tail(3))
display(silver_1d.tail(3))

--- SILVER FILES ---
.gitkeep
silver_market_4h.csv
silver_market_4h.json
silver_market_macro_sentiment_1d.csv
silver_market_macro_sentiment_1d.json
silver_market_sentiment_1h.csv
silver_market_sentiment_1h.json


,timestamp,open,high,low,close,volume,source,symbol,interval,sentiment_compound_mean,sentiment_ordinal_mean,sentiment_ordinal_sum,mentions
997,2026-03-11 03:00:00+00:00,69864.58,69921.09,69500.42,69553.72,0.71190,binanceus_ccxt,BTC/USDT,1h,-0.279125,-0.5,-2.0,4.0
998,2026-03-11 04:00:00+00:00,69568.90,70247.65,69535.97,70150.45,0.21527,binanceus_ccxt,BTC/USDT,1h,0.951200,2.0,2.0,1.0
999,2026-03-11 05:00:00+00:00,70164.40,70196.59,70116.29,70116.29,0.04977,binanceus_ccxt,BTC/USDT,1h,0.000000,0.0,0.0,1.0


,timestamp,open,high,low,close,volume,source,symbol,interval,sentiment_compound_mean,sentiment_ordinal_mean,sentiment_ordinal_sum,mentions
997,2026-03-09 00:00:00+00:00,65992.98,69519.89,65889.08,68374.92,24.43393,binanceus_ccxt,BTC/USDT,1d,0.00000,0.00,0.0,1.0
998,2026-03-10 00:00:00+00:00,68404.53,71774.18,68404.53,69937.15,45.54632,binanceus_ccxt,BTC/USDT,1d,0.11845,0.25,0.5,2.0
999,2026-03-11 00:00:00+00:00,69900.70,70247.65,69500.42,70116.29,2.24451,binanceus_ccxt,BTC/USDT,1d,-0.20300,-0.50,-0.5,1.0
